# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Score posts using author authority + content signals to rank them for 24h engagement.
Formula: score = 0.5*normalized_followers + 0.3*normalized_hashtags + 0.2*normalized_text_length

Reason codes:
R1 = HIGH_AUTHORITY: author_follower_count > 90th percentile
R2 = GOOD_HASHTAGS: hashtags_count between 3 and 8
R3 = LONG_FORM: text_length > 200 characters
R4 = LOW_SIGNAL: score < 20th percentile

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import os

df = pd.read_csv('../../data/starter.csv')

# Normalize features 0-100
df['norm_followers'] = (df['author_follower_count'] / df['author_follower_count'].max()) * 100
df['norm_hashtags'] = (df['hashtags_count'] / df['hashtags_count'].max()) * 100
df['norm_text'] = (df['text_length'] / df['text_length'].max()) * 100

# Baseline score
df['baseline_score'] = 0.5*df['norm_followers'] + 0.3*df['norm_hashtags'] + 0.2*df['norm_text']

# Reason codes
def get_reason(row):
    reasons = []
    if row['author_follower_count'] > df['author_follower_count'].quantile(0.9): reasons.append('R1')
    if 3 <= row['hashtags_count'] <= 8: reasons.append('R2')
    if row['text_length'] > 200: reasons.append('R3')
    if row['baseline_score'] < df['baseline_score'].quantile(0.2): reasons.append('R4')
    return ','.join(reasons) if reasons else 'R0'

df['reason_code'] = df.apply(get_reason, axis=1)

# Rank and save
df_sorted = df.sort_values('baseline_score', ascending=False)
os.makedirs('../../work/outputs', exist_ok=True)
df_sorted.to_csv('../../work/outputs/baseline_action_score.csv', index=False)

print("Saved to work/outputs/baseline_action_score.csv")
print(df_sorted[['baseline_score', 'reason_code']].head())

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = df_sorted.head(20)
for i, row in top20.iterrows():
    print(f"Rank {i+1}: Score={row['baseline_score']:.2f}, Reason={row['reason_code']}, Confidence=High if R1+R2 else Medium")

Top-20 Review:
Most top posts have R1 + R2 reason codes. 
What would make it wrong: If author buys followers later, or hashtags are spammy.
Confidence drops if content_type is new and not in training.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
weak = df_sorted.tail(10)
print("Weakest 10 scores:")
print(weak[['baseline_score', 'reason_code']])

# Leakage check
print("Leakage check: Score uses only pre-publish features. No engagement_score_24h used.")

Weak picks usually have R4. 
No leakage: All inputs available at decision time.

## Self-check

Before you submit, confirm each line honestly:

- [ yes] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/w04_baseline_score.ipynb` — then submit your repo URL on the card. Done.